# agent-decision-9b: snap decisions about agent tool calls

[`grounded-ai/agent-decision-9b`](https://huggingface.co/grounded-ai/agent-decision-9b) is a fine-tune of Cloudflare's [Clef-flash](https://huggingface.co/Cloudflare/clef-flash) decision model. Given the tools, the user's request and the agent's proposed tool calls, it answers typed questions (*are these calls correct?*, *what is wrong with them?*, *should the agent call a tool, ask, or decline?*) with a calibrated probability for every answer, in one forward pass and without generating text.

Use it anywhere an agent's step needs a fast, calibrated check: as a hook that vets tool calls before they run (like the grounding check in [`grounded_ai`](https://github.com/grounded-ai/grounded_ai)'s Claude Code plugin), as a guardrail that stops an agent from acting on a confidently wrong call, for labelling or filtering agent traces, or as the cheap first stage of an evaluation cascade where only low-confidence decisions go to an LLM judge.

This notebook:

1. shows a few rows of the test data,
2. scores the released Clef-flash on a sample of the test set,
3. loads agent-decision-9b and scores the same rows,
4. compares the two, and runs one decision of your own.

**Runtime:** a GPU with bfloat16 and about 24 GB: **L4**, A100 or H100 (Runtime → Change runtime type). The free T4 does not work: the model is 19 GB in bf16, and Qwen3.5's linear attention produces NaNs in float16.

In [ ]:
# Colab ships torchao 0.10, which recent PEFT refuses; we do not use it.
%pip uninstall -y -q torchao
%pip install -q "transformers>=5.10.2" "peft>=0.18" huggingface_hub pandas flash-linear-attention

In [ ]:
import json, random, sys
import pandas as pd
import torch
from huggingface_hub import hf_hub_download, snapshot_download

assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "needs a GPU with bfloat16 (L4, A100, H100); a T4 will not work"
print(torch.cuda.get_device_name())
pd.set_option("display.max_colwidth", 120)

NVIDIA L4


## 1. The data

The test set ([`grounded-ai/agent-decision-eval`](https://huggingface.co/datasets/grounded-ai/agent-decision-eval)) holds 1,139 rows from APIs never seen in training, with a fixed number of rows per kind of example. Every label is exact:

- **Tool calls** come from [xLAM function-calling](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k), whose calls were verified by execution. Half are shown as they are. The rest carry exactly one edit that makes them wrong: a tool swapped for one whose schema rejects the arguments, a parameter the tool does not define, a value the user typed removed or changed to one they did not type, or one of several calls dropped.
- **Call, ask or decline** comes from [NVIDIA When2Call](https://huggingface.co/datasets/nvidia/When2Call)'s curated answers.

In [ ]:
path = hf_hub_download("grounded-ai/agent-decision-eval", "test.jsonl", repo_type="dataset", revision="v1")
TEST = [json.loads(line) for line in open(path)]
kind = lambda r: f'{r["source"]}:{r["edit"] or r["kind"]}'
print(len(TEST), "test rows")
print(pd.Series([kind(r) for r in TEST]).value_counts().to_string())

1139 test rows
xlam:swapped_tool             150
xlam:correct                  150
xlam:undefined_parameter      150
xlam:changed_value            144
when2call:cannot_answer       126
when2call:tool_call           126
when2call:request_for_info    111
xlam:missing_value            100
xlam:dropped_call              82


In [ ]:
def show(row, max_chars=1400):
    state = row["state"] if isinstance(row["state"], str) else json.dumps(row["state"], ensure_ascii=False, indent=1)
    print("=" * 100)
    print(kind(row))
    print(state if len(state) <= max_chars else state[:max_chars] + "\n...")
    for qid, q in row["questions"].items():
        print(f'\nQ: {q["instructions"]}')
        if q.get("criteria"):
            print("   options:", list(q["criteria"]))
        print("   gold:", row["gold"][qid])


# One example of every kind, fixed up front (the first transcript-style row of each kind), not picked by result.
EXAMPLES = {k: next(r for r in TEST if kind(r) == k and r["style"] == "transcript") for k in sorted({kind(r) for r in TEST})}
for k in ("xlam:changed_value", "xlam:undefined_parameter", "xlam:correct", "when2call:request_for_info"):
    show(EXAMPLES[k])

xlam:changed_value
TOOLS:
{"name": "v1", "description": "Retrieves filtered observations from the Version 1 endpoint based on specified time and filters.", "parameters": {"time": {"description": "Select one of the following timeframes:", "type": "str", "default": "today"}, "filter1": {"description": "Can be a specific user, type, or tag (e.g., url, domain, ip, sha256, md5 for type, and phishing, ransomware, CobaltStrike for tag). Default is 'phishing'.", "type": "str, optional", "default": "phishing"}, "filter2": {"description": "Can be a specific user, type, or tag (e.g., url, domain, ip, sha256, md5 for type, and phishing, ransomware, CobaltStrike for tag). Default is 'url'.", "type": "str, optional", "default": "url"}}}
{"name": "query", "description": "Queries information about an IP address using the NetDetective API.", "parameters": {"ipaddress": {"description": "The IP address to use for the query. Defaults to None, which uses the IP address of the connecting client.", "type": "

## 2. The released Clef-flash, zero-shot

We score a sample of 20 rows per kind of example (180 rows) so the notebook runs in minutes, plus the nine example rows. On the full test set the released model gets **63.3%** of decisions right, and only **75%** of its answers given with 90%+ confidence are correct.

In [ ]:
base_path = snapshot_download("Cloudflare/clef-flash", revision="8b2e5fd17c09fd49bc2880b805d5323ec7ff4ff3")
sys.path.insert(0, base_path)
from joint_schema_model import collate_records, encode_record, load_release_model, systemone

model, processor = load_release_model(base_path, device="cuda")

rng = random.Random(0)
SAMPLE = []
for k in sorted({kind(r) for r in TEST}):
    group = [r for r in TEST if kind(r) == k]
    SAMPLE += rng.sample(group, min(20, len(group)))
print(len(SAMPLE), "rows in the sample")

180 rows in the sample


In [ ]:
def gold_key(question, gold):
    return ("true" if gold else "false") if question["type"] == "noul" else str(gold)


FAMILY_NAMES = {"tool_call_ok": "calls correct?", "tool_call_kind": "what is wrong?", "call_ask_decline": "call / ask / decline"}


@torch.inference_mode()
def score(rows, batch_size=8):
    """One row per question: the model's answer, its confidence, and whether it matches the gold answer."""
    rows = sorted(rows, key=lambda r: len(json.dumps(r["state"])))
    out = []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i : i + batch_size]
        encoded = [encode_record(processor.tokenizer, {"id": r["id"], "state": r["state"], "questions": r["questions"]}) for r in chunk]
        logits = model(collate_records(encoded, processor.tokenizer.pad_token_id, torch.device("cuda")))
        for row, enc, per_question in zip(chunk, encoded, logits):
            for q, z in zip(enc.questions, per_question):
                p = z.float().softmax(-1)
                answer = q.option_ids[int(p.argmax())]
                gold = gold_key(row["questions"][q.question_id], row["gold"][q.question_id])
                out.append({"id": row["id"], "question": FAMILY_NAMES[row["families"][q.question_id]], "kind": kind(row),
                            "gold": gold, "answer": answer, "confidence": float(p.max()), "correct": answer == gold})
    return pd.DataFrame(out)


base = score(SAMPLE)
base_examples = score(list(EXAMPLES.values()))
print(f'Clef-flash: {base["correct"].mean():.1%} correct; '
      f'{(base["confidence"] >= 0.9).mean():.0%} answered with 90%+ confidence, of which {base[base["confidence"] >= 0.9]["correct"].mean():.1%} correct')
base.groupby("question")["correct"].mean().map("{:.1%}".format)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Clef-flash: 69.0% correct; 37% answered with 90%+ confidence, of which 79.8% correct


,correct
question,
call / ask / decline,56.7%
calls correct?,76.7%
what is wrong?,66.1%


## 3. agent-decision-9b

The fine-tune is a LoRA adapter on the backbone plus a retrained, calibrated joint head. Loading it on top of Clef-flash takes three lines.

In [ ]:
from peft import PeftModel
from safetensors.torch import load_file

ours = snapshot_download("grounded-ai/agent-decision-9b")
model.language_model = PeftModel.from_pretrained(model.language_model, ours).merge_and_unload()
model.head.load_state_dict(load_file(f"{ours}/joint_head.safetensors"))  # calibrated head

tuned = score(SAMPLE)
tuned_examples = score(list(EXAMPLES.values()))
print(f'agent-decision-9b: {tuned["correct"].mean():.1%} correct; '
      f'{(tuned["confidence"] >= 0.9).mean():.0%} answered with 90%+ confidence, of which {tuned[tuned["confidence"] >= 0.9]["correct"].mean():.1%} correct')

agent-decision-9b: 94.1% correct; 90% answered with 90%+ confidence, of which 96.8% correct


## 4. Before and after, on the same rows

In [ ]:
def table(by):
    t = pd.DataFrame({"Clef-flash": base.groupby(by)["correct"].mean(), "agent-decision-9b": tuned.groupby(by)["correct"].mean()})
    t.loc["all"] = [base["correct"].mean(), tuned["correct"].mean()]
    return t.map("{:.1%}".format)


table("question")

,Clef-flash,agent-decision-9b
question,,
call / ask / decline,56.7%,88.3%
calls correct?,76.7%,96.7%
what is wrong?,66.1%,94.9%
all,69.0%,94.1%


In [ ]:
# "Are the calls correct?" by kind of call: which mistakes each model catches.
ok_base, ok_tuned = base[base["question"] == "calls correct?"], tuned[tuned["question"] == "calls correct?"]
pd.DataFrame({"Clef-flash": ok_base.groupby("kind")["correct"].mean(),
              "agent-decision-9b": ok_tuned.groupby("kind")["correct"].mean()}).map("{:.1%}".format)

,Clef-flash,agent-decision-9b
kind,,
xlam:changed_value,75.0%,95.0%
xlam:correct,100.0%,100.0%
xlam:dropped_call,95.0%,90.0%
xlam:missing_value,65.0%,95.0%
xlam:swapped_tool,100.0%,100.0%
xlam:undefined_parameter,25.0%,100.0%


In [ ]:
# One example of every kind, answered by both models (the four shown in section 1 are among them).
side = base_examples.merge(tuned_examples, on=["id", "question", "kind", "gold"], suffixes=(" (base)", " (ours)"))
side[["kind", "question", "gold", "answer (base)", "confidence (base)", "answer (ours)", "confidence (ours)"]].round(3)

,kind,question,gold,answer (base),confidence (base),answer (ours),confidence (ours)
0,xlam:missing_value,calls correct?,false,false,0.762,false,0.998
1,xlam:missing_value,what is wrong?,wrong_arguments,wrong_arguments,0.825,wrong_arguments,0.999
2,xlam:correct,calls correct?,true,true,0.914,true,0.999
3,xlam:correct,what is wrong?,correct,correct,0.784,correct,0.999
4,xlam:dropped_call,calls correct?,false,false,0.841,false,0.994
5,xlam:swapped_tool,calls correct?,false,false,0.973,false,0.999
6,xlam:undefined_parameter,calls correct?,false,true,0.930,false,0.999
7,xlam:changed_value,calls correct?,false,true,0.751,true,0.917
8,xlam:changed_value,what is wrong?,wrong_arguments,correct,0.536,correct,0.952
9,when2call:request_for_info,call / ask / decline,ask,call,0.936,call,0.962


## 5. Ask your own question

The model answers through Clef's SystemOne interface: a `state` (any JSON or text) and typed `questions`. Here the user asked for the account `guest`, and the agent called the login tool with `guest789`.

In [ ]:
tools = [{"name": "loginuser", "description": "Log in a user.",
          "parameters": {"username": {"type": "str"}, "password": {"type": "str"}}}]
response = systemone(model, processor, {
    "model": "agent-decision-9b",
    "state": {"tools": tools, "user": "Log in a user named 'guest' with the password 'guestpass'.",
              "assistant_tool_calls": [{"name": "loginuser", "arguments": {"username": "guest789", "password": "guestpass"}}]},
    "questions": {
        "call_ok": {"type": "noul", "instructions": "Are the assistant's tool calls correct and complete for the user's request?"},
        "call_kind": {"type": "choice", "instructions": "What, if anything, is wrong with the proposed tool calls?",
                      "criteria": {"correct": "the calls are right and do everything the user asked",
                                   "wrong_tool": "a call uses a tool that cannot do what was asked",
                                   "wrong_arguments": "the right tool is called with arguments that do not match the request or the tool",
                                   "incomplete": "the calls are right but part of the request is not done"}},
    },
})
print(json.dumps(response["answers"], indent=2))

{
  "call_ok": {
    "type": "noul",
    "noul": 0.0054
  },
  "call_kind": {
    "type": "choice",
    "choice": "wrong_arguments",
    "confidence": 0.9981,
    "probabilities": {
      "correct": 0.0014,
      "wrong_tool": 0.0003,
      "wrong_arguments": 0.9981,
      "incomplete": 0.0002
    }
  }
}


## Full test results

On all 1,139 test rows (from the training run, same rows for both models):

| Question | Clef-flash | agent-decision-9b |
|---|---|---|
| Are the calls correct? | 68.0% | **98.1%** |
| What is wrong with them? | 56.2% | **96.4%** |
| Call, ask or decline? | 60.6% | **88.2%** |
| All | 63.3% | **95.3%** |

91% of agent-decision-9b's answers come with 90%+ confidence, and 97.7% of those are right (Clef-flash: 37%, of which 75% are right). Calibration error drops from 0.166 to 0.014.

**Limitations.** Wrong calls in training and test are exact edits of correct calls: certain to be wrong, but possibly easier to spot than real agent mistakes. "Ask the user for missing information" is the weakest decision (81% here, 44% in an unseen markdown layout). A call with a parameter the tool does not define counts as wrong.

Model: [grounded-ai/agent-decision-9b](https://huggingface.co/grounded-ai/agent-decision-9b) · Data: [grounded-ai/agent-decision-eval](https://huggingface.co/datasets/grounded-ai/agent-decision-eval) · Base: [Cloudflare/clef-flash](https://huggingface.co/Cloudflare/clef-flash) · Training data: xLAM function-calling (Salesforce), When2Call (NVIDIA), both CC-BY-4.0.